In [50]:
import pandas as pd 
import numpy as np 
import seaborn as sns 
import matplotlib.pyplot as plt
 

In [51]:
df=pd.read_csv('car_price_prediction_.csv')
df.head()

,Car ID,Brand,Year,Engine Size,Fuel Type,Transmission,Mileage,Condition,Price,Model
0,1,Tesla,2016,2.3,Petrol,Manual,114832,New,26613.92,Model X
1,2,BMW,2018,4.4,Electric,Manual,143190,Used,14679.61,5 Series
2,3,Audi,2013,4.5,Electric,Manual,181601,New,44402.61,A4
3,4,Tesla,2011,4.1,Diesel,Automatic,68682,New,86374.33,Model Y
4,5,Ford,2009,2.6,Diesel,Manual,223009,Like New,73577.10,Mustang


In [52]:
df.isnull().sum()

Car ID          0
Brand           0
Year            0
Engine Size     0
Fuel Type       0
Transmission    0
Mileage         0
Condition       0
Price           0
Model           0
dtype: int64

In [53]:
df=df.drop(columns=['Car ID', 'Brand'])

In [54]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Year          2500 non-null   int64  
 1   Engine Size   2500 non-null   float64
 2   Fuel Type     2500 non-null   str    
 3   Transmission  2500 non-null   str    
 4   Mileage       2500 non-null   int64  
 5   Condition     2500 non-null   str    
 6   Price         2500 non-null   float64
 7   Model         2500 non-null   str    
dtypes: float64(2), int64(2), str(4)
memory usage: 214.9 KB


In [55]:
num_feature=[feature for feature in df.columns if df[feature].dtype!='str' ]
cat_feature=[feature for feature in df.columns if df[feature].dtype=='str' ]

In [56]:
discrete_feature=[feature for feature in df.columns if len(df[feature].unique())<=25]
continous_feature=[feature for feature in df.columns if feature not in discrete_feature]

In [57]:
df.head()

,Year,Engine Size,Fuel Type,Transmission,Mileage,Condition,Price,Model
0,2016,2.3,Petrol,Manual,114832,New,26613.92,Model X
1,2018,4.4,Electric,Manual,143190,Used,14679.61,5 Series
2,2013,4.5,Electric,Manual,181601,New,44402.61,A4
3,2011,4.1,Diesel,Automatic,68682,New,86374.33,Model Y
4,2009,2.6,Diesel,Manual,223009,Like New,73577.10,Mustang


In [58]:
x=df.drop('Price', axis=1)
y=df['Price']

In [59]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
x['Model']=le.fit_transform(df['Model'])

In [60]:
num_feature=x.select_dtypes(exclude='str').columns 
onehot_columns=['Fuel Type', 'Transmission', 'Condition']
label_encoder_columns=['Model']

In [61]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
numeric_transformer=StandardScaler()
oh_transformer=OneHotEncoder()
preprocessor=ColumnTransformer(
    [
        ("StandardScaler", numeric_transformer, num_feature),
        ("OneHotEncoder", oh_transformer, cat_feature)
    ], remainder='passthrough'
)

In [62]:
print(type(x))

<class 'pandas.DataFrame'>


In [63]:
x=preprocessor.fit_transform(x)
pd.DataFrame(x)

,0
0,<Compressed Sparse Row sparse matrix of dtype ...
1,<Compressed Sparse Row sparse matrix of dtype ...
2,<Compressed Sparse Row sparse matrix of dtype ...
3,<Compressed Sparse Row sparse matrix of dtype ...
4,<Compressed Sparse Row sparse matrix of dtype ...
...,...
2495,<Compressed Sparse Row sparse matrix of dtype ...
2496,<Compressed Sparse Row sparse matrix of dtype ...
2497,<Compressed Sparse Row sparse matrix of dtype ...
2498,<Compressed Sparse Row sparse matrix of dtype ...


In [64]:
x

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 20000 stored elements and shape (2500, 41)>

In [65]:
from sklearn.model_selection import train_test_split
x_train,x_test, y_train,y_test=train_test_split(x,y, test_size=0.3, random_state=10)

In [66]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.neighbors import KNeighborsRegressor


In [67]:
def evaluate_model(true, predicted):
    mae=mean_absolute_error(true, predicted)
    mse=mean_squared_error(true,predicted)
    r2_square=r2_score(true,predicted)
    rmse=np.sqrt(mean_squared_error(true, predicted))
    return mae, mse, r2_square, rmse


In [68]:
models={
    "LinearRegressioon": LinearRegression(),
    "Lasso": Lasso(),
    "Ridge": Ridge(),
    "DecisionTreeRegressor": DecisionTreeRegressor(),
    "KNeighborsRegressor": KNeighborsRegressor(),
    "RandomForestRegressor":RandomForestRegressor()
}

In [69]:
for i in range(len(list(models))):
    model=list(models.values())[i]
    model.fit(x_train, y_train)

In [71]:
y_train_pred=model.predict(x_train)
y_test_pred=model.predict(x_test)

In [72]:
mode_train_mae, mode_train_mse,mode_train_rmse,mode_train_r2_square=evaluate_model(y_train, y_train_pred)
mode_test_mae,mode_test_mse,mode_test_rmse,mode_test_r2_square=evaluate_model(y_test, y_test_pred)

In [74]:
print(list(models.keys())[i])

RandomForestRegressor


In [73]:
print("train model accuracy",mode_train_mae, mode_train_mse,mode_train_rmse,mode_train_r2_square )
print("test model accuracy", mode_test_mae,mode_test_mse,mode_test_rmse,mode_test_r2_square)

train model accuracy 9056.250600571428 114363308.87359631 0.8512122188312686 10694.07821523652
test model accuracy 23323.674609466663 746597200.231211 -0.08385374428037906 27323.93090737881
